# Notebook 05 — Combined Dengue and Weather Exploratory Data Analysis (EDA)

**Project:** RiskWeave — Climate-informed dengue risk forecasting and inspection planning for Sri Lanka

## Purpose
This notebook explores the existing combined historical dataset before dengue model development. It examines data structure and quality, dengue trends, district differences, seasonality, and associations between dengue cases and weather variables.

**Important:** This notebook does not train prediction models. It does not establish causation. It also does not use future information to create forecasting features.

### Input
`data/processed/dengue_weather_weekly_25_districts.csv`

### Outputs
EDA summary CSV files and charts will be saved under `outputs/eda/`.

### Methodological notes
- Preserve the original combined dataset; do not overwrite it.
- The observation unit is a district × dengue reporting period.
- Reporting periods are not assumed to be Monday–Sunday calendar weeks.
- Ampara/Kalmune aggregation is inherited from the upstream dataset and must be verified as an approved surveillance-area mapping.
- The weather source in the collected data is Open-Meteo ERA5, not NASA POWER. Document this source difference in the project report.
- Associations found here are exploratory, not proof that weather caused dengue cases.


## 1. Import libraries and configure paths

Run this notebook with the project folder as the working directory, or from the `notebooks` folder. The path setup below handles both cases.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Detect project root whether Jupyter starts in RiskWeave/ or RiskWeave/notebooks/
cwd = Path.cwd()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the RiskWeave project root. "
        "Start Jupyter from the RiskWeave folder or its notebooks folder."
    )

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "dengue_weather_weekly_25_districts.csv"
EDA_DIR = PROJECT_ROOT / "outputs" / "eda"
EDA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Input file:", DATA_PATH)
print("EDA output folder:", EDA_DIR)
print("Input exists:", DATA_PATH.exists())

## 2. Load the combined dataset

We load the dataset without changing the source file. The original column names are displayed so we can confirm the actual schema before analysis.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Main dataset was not found: {DATA_PATH}\n"
        "Check that Notebook 04 completed and saved the combined CSV."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())
display(df.info())

## 3. Standardize date types and inspect required columns

This step converts reporting-period dates into pandas datetime values. It does not change the date values themselves.

The combined dataset is expected to contain `area`, `week_start`, `week_end`, and `dengue_cases`. If your file uses different names, stop and inspect the column list rather than silently renaming columns.

In [ ]:
required_cols = ["area", "week_start", "week_end", "dengue_cases"]
missing_required = [col for col in required_cols if col not in df.columns]

if missing_required:
    raise ValueError(
        f"Missing required columns: {missing_required}. "
        "Review the actual column names printed in Section 2."
    )

df["week_start"] = pd.to_datetime(df["week_start"], errors="coerce")
df["week_end"] = pd.to_datetime(df["week_end"], errors="coerce")
df["dengue_cases"] = pd.to_numeric(df["dengue_cases"], errors="coerce")

print("Date range:", df["week_start"].min(), "to", df["week_end"].max())
print("Number of reporting areas:", df["area"].nunique())
print("Areas:", sorted(df["area"].dropna().unique()))
print("Reporting periods:", df[["week_start", "week_end"]].drop_duplicates().shape[0])

display(df[required_cols].head())

## 4. Data quality checks

We check missing values, duplicate district-period keys, invalid date intervals, negative case counts, and reporting-period lengths.

A flagged row is investigated; it is not automatically deleted.

In [ ]:
key_cols = ["area", "week_start", "week_end"]

quality_summary = pd.DataFrame({
    "check": [
        "rows",
        "columns",
        "missing area",
        "missing week_start",
        "missing week_end",
        "missing dengue_cases",
        "duplicate area-reporting-period keys",
        "invalid date intervals (week_end before week_start)",
        "negative dengue case values",
    ],
    "value": [
        len(df),
        len(df.columns),
        int(df["area"].isna().sum()),
        int(df["week_start"].isna().sum()),
        int(df["week_end"].isna().sum()),
        int(df["dengue_cases"].isna().sum()),
        int(df.duplicated(key_cols).sum()),
        int((df["week_end"] < df["week_start"]).sum()),
        int((df["dengue_cases"] < 0).sum()),
    ]
})

display(quality_summary)
quality_summary.to_csv(EDA_DIR / "eda_data_quality_summary.csv", index=False)

df["reporting_period_days_check"] = (
    df["week_end"] - df["week_start"]
).dt.days + 1

display(
    df["reporting_period_days_check"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("reporting_period_days")
    .reset_index(name="row_count")
)

period_length_summary = (
    df.groupby("reporting_period_days_check", dropna=False)
      .size()
      .reset_index(name="row_count")
      .sort_values("reporting_period_days_check")
)
period_length_summary.to_csv(EDA_DIR / "eda_reporting_period_lengths.csv", index=False)

## 5. Dengue cases: overall distribution

Dengue case counts are count data and may be highly skewed. We inspect summary statistics and a histogram. The histogram is descriptive only; it does not determine a model family by itself.

In [ ]:
display(df["dengue_cases"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

fig, ax = plt.subplots(figsize=(9, 5))
valid_cases = df["dengue_cases"].dropna()
ax.hist(valid_cases, bins=40)
ax.set_title("Distribution of weekly dengue cases per reporting area")
ax.set_xlabel("Dengue cases")
ax.set_ylabel("Number of district-reporting-period observations")
fig.tight_layout()
fig.savefig(EDA_DIR / "dengue_cases_distribution.png", dpi=150)
plt.show()

## 6. Weekly dengue trends over time

The dataset is district-level. To show the national total per reporting period, we sum cases across the included reporting areas for each exact `week_start` and `week_end`.

**Caution:** This is the sum of the 25 reporting areas represented in the processed dataset. It should not be described as an independently verified national total until the Ampara/Kalmune reporting-area mapping is confirmed.

In [ ]:
national_period = (
    df.groupby(["week_start", "week_end"], as_index=False)
      .agg(total_dengue_cases=("dengue_cases", "sum"))
      .sort_values("week_start")
)

display(national_period.head())
national_period.to_csv(EDA_DIR / "eda_total_dengue_by_reporting_period.csv", index=False)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(national_period["week_start"], national_period["total_dengue_cases"])
ax.set_title("Dengue cases across included reporting areas over time")
ax.set_xlabel("Reporting period start")
ax.set_ylabel("Total dengue cases")
fig.tight_layout()
fig.savefig(EDA_DIR / "dengue_overall_time_trend.png", dpi=150)
plt.show()

## 7. District-level dengue comparison

We compare total, mean, median, maximum and variability of weekly cases by reporting area. High totals can reflect population size as well as disease burden; these summaries are not population-adjusted incidence rates.

In [ ]:
district_summary = (
    df.groupby("area")["dengue_cases"]
      .agg(
          total_cases="sum",
          mean_weekly_cases="mean",
          median_weekly_cases="median",
          maximum_weekly_cases="max",
          standard_deviation="std",
          reporting_periods="count"
      )
      .reset_index()
      .sort_values("total_cases", ascending=False)
)

display(district_summary)
district_summary.to_csv(EDA_DIR / "eda_district_dengue_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 7))
plot_data = district_summary.sort_values("total_cases")
ax.barh(plot_data["area"], plot_data["total_cases"])
ax.set_title("Total recorded dengue cases by reporting area")
ax.set_xlabel("Total dengue cases across the study period")
ax.set_ylabel("Reporting area")
fig.tight_layout()
fig.savefig(EDA_DIR / "dengue_total_by_area.png", dpi=150)
plt.show()

## 8. Seasonal patterns

We calculate the calendar month from each reporting period's start date and summarize dengue cases by month. Because reporting periods can cross month boundaries, this is an approximate seasonal description based on `week_start`, not a day-weighted monthly case allocation.

For formal seasonal modelling, we will use carefully defined week-of-year or cyclic time features and preserve chronological validation.

In [ ]:
df["start_month"] = df["week_start"].dt.month
df["start_year"] = df["week_start"].dt.year

monthly_seasonality = (
    df.groupby("start_month")["dengue_cases"]
      .agg(
          mean_cases="mean",
          median_cases="median",
          total_cases="sum",
          observations="count"
      )
      .reindex(range(1, 13))
      .reset_index()
)

display(monthly_seasonality)
monthly_seasonality.to_csv(EDA_DIR / "eda_dengue_by_start_month.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(monthly_seasonality["start_month"], monthly_seasonality["mean_cases"], marker="o")
ax.set_xticks(range(1, 13))
ax.set_xlabel("Month of reporting-period start")
ax.set_ylabel("Mean dengue cases per area-period")
ax.set_title("Average dengue cases by month")
fig.tight_layout()
fig.savefig(EDA_DIR / "dengue_monthly_seasonality.png", dpi=150)
plt.show()

## 9. Yearly dengue patterns

This summarizes the annual number of recorded cases across the included reporting areas. The source ends in December 2025, so the final year may cover only part of a calendar year depending on the exact source coverage.

In [ ]:
yearly_summary = (
    df.groupby("start_year")["dengue_cases"]
      .agg(
          total_cases="sum",
          mean_cases_per_area_period="mean",
          reporting_area_periods="count"
      )
      .reset_index()
)

display(yearly_summary)
yearly_summary.to_csv(EDA_DIR / "eda_dengue_by_year.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(yearly_summary["start_year"], yearly_summary["total_cases"], marker="o")
ax.set_xlabel("Year of reporting-period start")
ax.set_ylabel("Total recorded dengue cases")
ax.set_title("Yearly dengue cases across included reporting areas")
fig.tight_layout()
fig.savefig(EDA_DIR / "dengue_yearly_trend.png", dpi=150)
plt.show()

## 10. Weather variables available for analysis

We identify numeric weather columns from the combined dataset. The notebook prints the candidates so that we can inspect the actual schema and avoid assuming a column exists.

We exclude identifiers, dates, dengue target, and helper fields from the weather-variable list.

In [ ]:
exclude_from_weather = {
    "dengue_cases",
    "year",
    "week",
    "start_month",
    "start_year",
    "reporting_period_days_check",
}

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
weather_numeric_cols = [
    col for col in numeric_cols
    if col not in exclude_from_weather
    and not col.lower().endswith("_cases")
]

print("Numeric columns:", numeric_cols)
print("\nCandidate numeric weather/features:")
print(weather_numeric_cols)

display(df[weather_numeric_cols].describe().T if weather_numeric_cols else
        pd.DataFrame({"message": ["No numeric weather columns were detected."]}))

## 11. Exploratory correlations between dengue and weather

We calculate Pearson and Spearman correlations between dengue cases and candidate numeric weather features.

- Pearson measures linear association.
- Spearman measures rank-based monotonic association and is less sensitive to some distribution shapes.
- These are exploratory, pooled correlations. They do not establish causation and can be affected by seasonality, district differences, autocorrelation and outliers.
- Correlations are not a feature-selection rule on their own.

In [ ]:
correlation_rows = []

for col in weather_numeric_cols:
    pair = df[["dengue_cases", col]].dropna()
    if len(pair) >= 3 and pair["dengue_cases"].nunique() > 1 and pair[col].nunique() > 1:
        pearson_value = pair["dengue_cases"].corr(pair[col], method="pearson")
        spearman_value = pair["dengue_cases"].corr(pair[col], method="spearman")
    else:
        pearson_value = np.nan
        spearman_value = np.nan

    correlation_rows.append({
        "weather_feature": col,
        "pearson_correlation": pearson_value,
        "spearman_correlation": spearman_value,
        "valid_observations": len(pair)
    })

correlation_summary = pd.DataFrame(correlation_rows)
if not correlation_summary.empty:
    correlation_summary = correlation_summary.sort_values(
        "spearman_correlation", key=lambda s: s.abs(), ascending=False
    )

display(correlation_summary)
correlation_summary.to_csv(EDA_DIR / "eda_dengue_weather_correlations.csv", index=False)

## 12. District-specific weather–dengue correlations

Pooled correlations can hide differences between districts. This section calculates within-area Spearman correlations where enough valid observations exist.

These results are exploratory and should be interpreted cautiously, especially where observations are serially correlated.

In [ ]:
district_corr_rows = []

for area, area_df in df.groupby("area"):
    for col in weather_numeric_cols:
        pair = area_df[["dengue_cases", col]].dropna()
        if len(pair) >= 20 and pair["dengue_cases"].nunique() > 1 and pair[col].nunique() > 1:
            corr_value = pair["dengue_cases"].corr(pair[col], method="spearman")
        else:
            corr_value = np.nan

        district_corr_rows.append({
            "area": area,
            "weather_feature": col,
            "spearman_correlation": corr_value,
            "valid_observations": len(pair)
        })

district_weather_correlations = pd.DataFrame(district_corr_rows)
display(district_weather_correlations.head(20))
district_weather_correlations.to_csv(
    EDA_DIR / "eda_district_weather_correlations.csv", index=False
)

## 13. Inspect possible target-data issues and extreme observations

We list the highest dengue case observations for manual review. These may represent genuine outbreaks, reporting changes or data issues. Do not remove them simply because they are extreme.

If population denominators are later obtained from a trustworthy source, we can consider incidence rates. They are not available in this combined dataset as currently described.

In [ ]:
top_case_rows = (
    df.sort_values("dengue_cases", ascending=False)
      [["area", "week_start", "week_end", "dengue_cases"]]
      .head(30)
)

display(top_case_rows)
top_case_rows.to_csv(EDA_DIR / "eda_highest_dengue_observations_review.csv", index=False)

## 14. Prepare a compact EDA findings checklist

Use the printed outputs and saved charts to complete this checklist. Do not write conclusions until you have run the notebook and inspected the actual results.

1. Are there missing values or duplicate district-period records?
2. Are all reporting periods the expected length, and have exceptional 6- or 8-day periods been retained correctly?
3. Which reporting areas have the largest total and average dengue counts?
4. Do the time-series charts show visible peaks, long-term changes or unusual reporting gaps?
5. Are there seasonal patterns in the month summary?
6. Which weather variables show exploratory associations with dengue?
7. Do those associations differ between reporting areas?
8. What limitations must be considered before modelling?

### Important modelling decisions still to make
- Confirm whether Ampara and Kalmune can be combined for the intended surveillance geography.
- Decide the forecast horizon and prediction origin (for example, predict the next dengue reporting period using information available up to the previous period).
- Define chronological train/validation/test periods.
- Build lagged dengue and weather features using past data only.
- For operational-style evaluation, use weather forecasts available at each historical forecast origin, not actual weather observed during the target period.
- Compare models against simple baselines.


## 15. Save a record of the dataset used

This small manifest records the input file name, row/column counts and coverage dates for reproducibility.

In [ ]:
manifest = pd.DataFrame([{
    "input_file": str(DATA_PATH),
    "rows": len(df),
    "columns": len(df.columns),
    "number_of_areas": df["area"].nunique(),
    "first_reporting_period_start": df["week_start"].min(),
    "last_reporting_period_end": df["week_end"].max(),
    "notebook": "05_combined_dengue_weather_eda.ipynb"
}])

display(manifest)
manifest.to_csv(EDA_DIR / "eda_run_manifest.csv", index=False)

print("\nEDA notebook completed.")
print("Outputs saved to:", EDA_DIR)